# DAY 9 - Statistics for Data Analytics

**Statistics is the language of data.**

Every interview asks statistical questions:
- What is mean vs median?
- What is standard deviation?
- What is correlation?
- What is a normal distribution?
- What is a p-value?

---

## Topics Covered
1. Types of Data
2. Measures of Central Tendency (Mean, Median, Mode)
3. Measures of Spread (Range, Variance, Std Dev, IQR)
4. Skewness and Kurtosis
5. Percentiles and Quartiles
6. Probability Basics
7. Normal Distribution (Bell Curve)
8. Z-Score
9. Correlation
10. Hypothesis Testing (t-test, chi-square)
11. AB Testing (real-world)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import ttest_ind, chi2_contingency, norm

plt.rcParams['figure.figsize'] = (10, 5)
sns.set_theme(style='whitegrid')
print('Libraries loaded!')

---
## 1. Types of Data

```
Data
├── Quantitative (Numbers)
│   ├── Discrete   → Countable (students: 30, 31, 32)
│   └── Continuous → Any value (height: 5.7, 5.71, 5.712)
└── Qualitative (Categories)
    ├── Nominal    → No order (city: Delhi, Mumbai)
    └── Ordinal    → Ordered (rating: Poor < Good < Excellent)
```

In [ ]:
examples = pd.DataFrame({
    'Data': ['Number of students in class', 'Student height in cm', 'City names', 'Grade (A/B/C/D)'],
    'Type': ['Discrete', 'Continuous', 'Nominal', 'Ordinal'],
    'Category': ['Quantitative', 'Quantitative', 'Qualitative', 'Qualitative']
})
print('Examples of Data Types:')
print(examples.to_string(index=False))

---
## 2. Measures of Central Tendency

Answers: "Where is the center of data?"

In [ ]:
marks = [72, 65, 88, 55, 92, 72, 78, 83, 65, 72, 95, 60]

mean   = np.mean(marks)
median = np.median(marks)
mode   = stats.mode(marks, keepdims=True)

print(f'Data: {sorted(marks)}')
print(f'\nMean   (Average)         : {mean:.2f}')
print(f'Median (Middle Value)    : {median}')
print(f'Mode   (Most Frequent)   : {mode.mode[0]} (appears {mode.count[0]} times)')

print('\n--- When to use what? ---')
print('Mean   → Symmetric data, no outliers (example: test scores)')
print('Median → Skewed data or outliers (example: salary, house price)')
print('Mode   → Categorical data (example: most popular product)')

In [ ]:
# Effect of outlier on Mean vs Median
normal_salaries  = [40000, 45000, 50000, 52000, 48000, 46000, 51000]
ceo_salary       = 5000000
with_ceo         = normal_salaries + [ceo_salary]

print('Without CEO salary:')
print(f'  Mean   = Rs {np.mean(normal_salaries):,.0f}')
print(f'  Median = Rs {np.median(normal_salaries):,.0f}')

print('\nWith CEO salary (outlier):')
print(f'  Mean   = Rs {np.mean(with_ceo):,.0f}   ← DISTORTED by outlier')
print(f'  Median = Rs {np.median(with_ceo):,.0f}   ← Not affected')
print('\n→ Use Median for salary analysis!')

---
## 3. Measures of Spread

Answers: "How spread out is the data?"

In [ ]:
data = np.array([55, 60, 65, 70, 72, 75, 78, 82, 85, 88, 92, 95])

print(f'Data: {data}')
print()
print(f'Range       = Max - Min = {data.max()} - {data.min()} = {data.max()-data.min()}')
print(f'Variance    = {np.var(data):.2f}  (average squared deviation)')
print(f'Std Dev     = {np.std(data):.2f}  (square root of variance)')
print()
Q1 = np.percentile(data, 25)
Q3 = np.percentile(data, 75)
IQR = Q3 - Q1
print(f'Q1 (25th %) = {Q1}')
print(f'Q3 (75th %) = {Q3}')
print(f'IQR         = Q3 - Q1 = {IQR}')
print()
print('--- Interpretation ---')
print(f'Low std dev  = data is clustered close to mean')
print(f'High std dev = data is widely spread')

In [ ]:
# Visual: Two classes same mean, different spread
np.random.seed(42)
class_a = np.random.normal(75, 5, 200)   # mean=75, std=5 (consistent)
class_b = np.random.normal(75, 20, 200)  # mean=75, std=20 (variable)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, data, label, color in [
    (axes[0], class_a, f'Class A\nMean={class_a.mean():.1f}, Std={class_a.std():.1f}', 'steelblue'),
    (axes[1], class_b, f'Class B\nMean={class_b.mean():.1f}, Std={class_b.std():.1f}', 'tomato')
]:
    ax.hist(data, bins=25, color=color, edgecolor='white', alpha=0.8)
    ax.axvline(data.mean(), color='black', linestyle='--', lw=2, label='Mean')
    ax.set_title(label, fontsize=13)
    ax.set_xlabel('Score')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle('Same Mean, Different Spread', fontsize=16)
plt.tight_layout()
plt.show()

---
## 4. Skewness and Kurtosis

**Skewness** = How asymmetric the distribution is

- Skew = 0 → Symmetric (normal)
- Skew > 0 → Right skewed (tail on right, mean > median)
- Skew < 0 → Left skewed (tail on left, mean < median)

In [ ]:
np.random.seed(10)
symmetric   = np.random.normal(50, 10, 500)
right_skew  = np.random.exponential(scale=20, size=500) + 10
left_skew   = 100 - np.random.exponential(scale=20, size=500)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, data, title, color in [
    (axes[0], symmetric,  f'Symmetric\nskew={stats.skew(symmetric):.2f}', 'steelblue'),
    (axes[1], right_skew, f'Right Skewed\nskew={stats.skew(right_skew):.2f}', 'tomato'),
    (axes[2], left_skew,  f'Left Skewed\nskew={stats.skew(left_skew):.2f}', 'goldenrod')
]:
    ax.hist(data, bins=30, color=color, edgecolor='white', alpha=0.8)
    ax.axvline(np.mean(data), color='red', linestyle='--', lw=2, label='Mean')
    ax.axvline(np.median(data), color='blue', linestyle='-', lw=2, label='Median')
    ax.set_title(title, fontsize=13)
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle('Types of Skewness', fontsize=16)
plt.tight_layout()
plt.show()

print('Real-world examples:')
print('Right skewed → Salaries, Home prices, Company revenue')
print('Left skewed  → Exam scores (when exam is easy)')
print('Symmetric    → Height, IQ scores, measurement errors')

---
## 5. Percentiles and Quartiles

In [ ]:
salaries = np.random.normal(60000, 15000, 1000).astype(int)

percentiles = [10, 25, 50, 75, 90, 95, 99]
print('Salary Percentiles:')
for p in percentiles:
    value = np.percentile(salaries, p)
    print(f'  {p:>3}th percentile = Rs {value:>8,}')

print(f'\nIf your salary is Rs 75,000:')
your_salary = 75000
your_percentile = stats.percentileofscore(salaries, your_salary)
print(f'You are at the {your_percentile:.1f}th percentile')
print(f'You earn more than {your_percentile:.1f}% of employees')

---
## 6. Normal Distribution (Bell Curve)

Most important distribution in statistics.

**Empirical Rule (68-95-99.7):**
- 68% of data falls within 1 standard deviation of mean
- 95% within 2 standard deviations
- 99.7% within 3 standard deviations

In [ ]:
mu, sigma = 170, 10  # Height: mean=170cm, std=10cm
x = np.linspace(130, 210, 300)
y = norm.pdf(x, mu, sigma)

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(x, y, 'k-', linewidth=2, label='Normal Distribution')

# Shade regions
for n_std, color, alpha, label in [
    (3, 'lightblue',  0.8, '99.7% (±3σ)'),
    (2, 'steelblue',  0.6, '95.0% (±2σ)'),
    (1, 'darkblue',   0.4, '68.0% (±1σ)')
]:
    x_fill = np.linspace(mu - n_std*sigma, mu + n_std*sigma, 200)
    ax.fill_between(x_fill, norm.pdf(x_fill, mu, sigma), alpha=alpha, color=color, label=label)

ax.axvline(mu, color='red', linestyle='--', lw=2, label=f'Mean={mu}')
for n in [1, 2, 3]:
    ax.axvline(mu + n*sigma, color='gray', linestyle=':', lw=1)
    ax.axvline(mu - n*sigma, color='gray', linestyle=':', lw=1)
    ax.text(mu + n*sigma, 0.001, f'+{n}σ', ha='center', fontsize=9)
    ax.text(mu - n*sigma, 0.001, f'-{n}σ', ha='center', fontsize=9)

ax.set_title('Normal Distribution — Height Example (Mean=170cm, Std=10cm)', fontsize=14)
ax.set_xlabel('Height (cm)')
ax.set_ylabel('Probability Density')
ax.legend(loc='upper right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 7. Z-Score (Standardization)

**Z-score tells you how many standard deviations a value is from the mean.**

Formula: `Z = (X - mean) / std`

In [ ]:
# Class test scores
scores = np.array([55, 70, 82, 65, 90, 78, 48, 88, 72, 60])

mean_score = np.mean(scores)
std_score  = np.std(scores)
z_scores   = (scores - mean_score) / std_score

df_z = pd.DataFrame({
    'Score': scores,
    'Z-Score': z_scores.round(2),
    'Interpretation': []
})

interp = []
for z in z_scores:
    if z > 2: interp.append('Very High (above 95%)')
    elif z > 1: interp.append('High (above 84%)')
    elif z > 0: interp.append('Above Average')
    elif z > -1: interp.append('Below Average')
    elif z > -2: interp.append('Low (below 16%)')
    else: interp.append('Very Low (below 2%)')

df_z['Interpretation'] = interp
print(f'Mean = {mean_score:.1f}, Std Dev = {std_score:.2f}')
print()
print(df_z)

print(f'\nThe student who scored 90:')
z = (90 - mean_score) / std_score
print(f'  Z-score = {z:.2f}')
print(f'  They scored better than {norm.cdf(z)*100:.1f}% of the class')

---
## 8. Correlation

**How strongly two variables move together.**

- Pearson r = -1 → Perfect negative correlation
- Pearson r = 0  → No correlation
- Pearson r = +1 → Perfect positive correlation

In [ ]:
np.random.seed(42)
hours_studied = np.random.uniform(1, 10, 80)
exam_score    = 40 + hours_studied * 5 + np.random.normal(0, 5, 80)

correlation_r, p_value = stats.pearsonr(hours_studied, exam_score)

print(f'Pearson Correlation (r)  = {correlation_r:.4f}')
print(f'P-value                  = {p_value:.6f}')
if p_value < 0.05:
    print(f'Result: STATISTICALLY SIGNIFICANT correlation')
print(f'\nInterpretation: Study hours and exam scores have')
print(f'a {"strong" if abs(correlation_r) > 0.7 else "moderate"} positive correlation.')

plt.figure(figsize=(9, 5))
plt.scatter(hours_studied, exam_score, alpha=0.6, color='steelblue', s=60)
z = np.polyfit(hours_studied, exam_score, 1)
p = np.poly1d(z)
plt.plot(np.sort(hours_studied), p(np.sort(hours_studied)), 'r-', lw=2, label=f'r = {correlation_r:.2f}')
plt.title('Study Hours vs Exam Score', fontsize=15)
plt.xlabel('Hours Studied')
plt.ylabel('Exam Score')
plt.legend(fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 9. Hypothesis Testing

**Hypothesis testing answers: Is the difference real or just by chance?**

```
H0 (Null Hypothesis)      = No difference / No effect
H1 (Alternative Hypothesis) = There IS a difference

If p-value < 0.05 → Reject H0 (difference is real)
If p-value >= 0.05 → Cannot reject H0
```

In [ ]:
# T-Test: Do two groups have the same mean?
# Question: Is there a significant salary difference between IT and HR?
np.random.seed(42)
it_salary  = np.random.normal(85000, 12000, 50)
hr_salary  = np.random.normal(65000, 10000, 50)

print('IT Department:', f'Mean = Rs {it_salary.mean():,.0f}, Std = Rs {it_salary.std():,.0f}')
print('HR Department:', f'Mean = Rs {hr_salary.mean():,.0f}, Std = Rs {hr_salary.std():,.0f}')

t_stat, p_value = ttest_ind(it_salary, hr_salary)
print(f'\nTwo-Sample T-Test:')
print(f'  t-statistic = {t_stat:.4f}')
print(f'  p-value     = {p_value:.6f}')

if p_value < 0.05:
    print(f'\nResult: p < 0.05 → REJECT H0')
    print('Conclusion: Salary difference between IT and HR is STATISTICALLY SIGNIFICANT.')
else:
    print(f'\nResult: p >= 0.05 → CANNOT reject H0')
    print('Conclusion: No significant salary difference found.')

In [ ]:
# Chi-Square Test: Are two categorical variables related?
# Question: Is there a relationship between department and performance?

contingency = pd.DataFrame({
    'Excellent': [20, 10, 15],
    'Good':      [25, 20, 20],
    'Average':   [15, 25, 15],
    'Poor':      [ 5, 10,  5]
}, index=['IT', 'HR', 'Finance'])

print('Observed Frequency Table:')
print(contingency)

chi2, p, dof, expected = chi2_contingency(contingency)
print(f'\nChi-Square Test:')
print(f'  Chi2 statistic = {chi2:.4f}')
print(f'  p-value        = {p:.4f}')
print(f'  Degrees of freedom = {dof}')

if p < 0.05:
    print('\nResult: p < 0.05 → REJECT H0')
    print('Conclusion: Department and Performance are RELATED.')
else:
    print('\nResult: p >= 0.05 → Cannot reject H0')
    print('Conclusion: No significant relationship found.')

---
## 10. A/B Testing (Real-World Application)

**A/B testing = Comparing two versions to find the better one.**

Used by: Google, Amazon, Facebook, every e-commerce company.

In [ ]:
# A/B Test: Does new website button color improve conversion?
np.random.seed(42)

version_a_conversions = np.random.binomial(1, 0.12, 500)  # Old: 12% conversion
version_b_conversions = np.random.binomial(1, 0.15, 500)  # New: 15% conversion

rate_a = version_a_conversions.mean()
rate_b = version_b_conversions.mean()

print('=== A/B Test: Website Button Color ===')
print(f'Version A (Blue):  {rate_a*100:.1f}% conversion ({version_a_conversions.sum()}/500)')
print(f'Version B (Green): {rate_b*100:.1f}% conversion ({version_b_conversions.sum()}/500)')
print(f'Uplift: +{(rate_b - rate_a)*100:.1f}%')

# T-test to check if significant
t_stat, p_value = ttest_ind(version_a_conversions, version_b_conversions)
print(f'\nStatistical Test:')
print(f'  p-value = {p_value:.4f}')

if p_value < 0.05:
    print('  Result: SIGNIFICANT → Version B wins! Ship the green button.')
else:
    print('  Result: NOT significant → No clear winner yet. Need more data.')

# Business impact
monthly_visitors = 100000
current_revenue_per_conversion = 5000
print(f'\nBusiness Impact (100K monthly visitors):')
print(f'  Current revenue: Rs {monthly_visitors * rate_a * current_revenue_per_conversion:>12,.0f}')
print(f'  With new button: Rs {monthly_visitors * rate_b * current_revenue_per_conversion:>12,.0f}')
print(f'  Monthly uplift:  Rs {monthly_visitors * (rate_b-rate_a) * current_revenue_per_conversion:>12,.0f}')

---
## Statistics Quick Reference

| Concept | Formula | When to Use |
|---------|---------|-------------|
| Mean | Sum/Count | Symmetric data |
| Median | Middle value | Skewed data, outliers |
| Mode | Most frequent | Categorical data |
| Std Dev | √Variance | Spread of data |
| IQR | Q3 - Q1 | Outlier detection |
| Z-Score | (X-μ)/σ | How extreme a value is |
| Pearson r | Corr formula | Linear relationship |
| T-test | Means comparison | 2-group numeric |
| Chi-square | Frequency table | 2 categorical variables |

---
## Key Numbers to Remember

- p < 0.05 → Result is statistically significant
- |r| > 0.7 → Strong correlation
- |r| 0.3-0.7 → Moderate correlation
- |r| < 0.3 → Weak correlation
- Z > 3 or Z < -3 → Likely an outlier

---
## Homework

1. Create a dataset of 50 student marks and compute all measures
2. Check if the distribution is normal or skewed
3. Compare marks of two classes using t-test
4. Compute correlation between study hours and marks
5. Conduct an A/B test on two different teaching methods